# Cohort Enrolment

Builds the enrolment funnel, summarizes discharge destinations, and compares included and excluded stays.


In [ ]:
# Parameters

# project_root receives generated figures. data_dir contains the flat CSV files written by the
# Create Analysis Dataset notebook. Tests override both with temporary paths.
project_root = "../"
data_dir = "../data"

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import SVG, display

from algo_vs_doc import (
    clustered_inference,
    cohorts,
    enrolment_plots,
    local_time,
    prediction_data,
)
from algo_vs_doc.scoring.final_assembly import (
    OUTCOME_ELIGIBILITY_STEPS,
    apply_enrolment_funnel,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

figures_folder = os.path.join(project_root, "figures")
outputs_folder = os.path.join(project_root, "results", "summaries")
os.makedirs(outputs_folder, exist_ok=True)

## Load the cohorts

Load and validate both hospital CSV files and construct the pooled cohort.


In [ ]:
enrolment_cohorts = prediction_data.load_cohorts(data_dir)

pd.DataFrame(
    [
        {"cohort": name, "icu_stays": len(frame)}
        for name, frame in enrolment_cohorts.items()
    ]
)

## Enrolment funnel

Apply two ordered phases and report every exclusion separately. The clinical eligibility steps retain discharges to general wards and higher-dependency units. The final two steps require both predictions for the paired comparison.


In [ ]:
outcome_eligible = {}
enrolled = {}
reports = {}

for name, frame in enrolment_cohorts.items():
    outcome_eligible[name], _ = apply_enrolment_funnel(
        frame, steps=OUTCOME_ELIGIBILITY_STEPS
    )
    enrolled[name], reports[name] = apply_enrolment_funnel(frame)

prediction_availability = pd.concat(
    [
        frame.assign(
            doc_prediction_available=frame["doc_predicted_at"].notna(),
            algo_prediction_available=frame["algo_predicted_at_doc_prediction"].notna(),
        )
        .assign(
            both_predictions_available=lambda df: (
                df["doc_prediction_available"] & df["algo_prediction_available"]
            )
        )
        .groupby("discharge_care_level", dropna=False)
        .agg(
            stays=("icu_stay_id", "size"),
            physician_predictions=("doc_prediction_available", "sum"),
            algorithm_predictions=("algo_prediction_available", "sum"),
            paired_predictions=("both_predictions_available", "sum"),
        )
        .assign(cohort=name)
        .reset_index()
        for name, frame in outcome_eligible.items()
    ],
    ignore_index=True,
)
display(
    prediction_availability[
        [
            "cohort",
            "discharge_care_level",
            "stays",
            "physician_predictions",
            "algorithm_predictions",
            "paired_predictions",
        ]
    ]
)

# one row per step, one column per cohort
print(
    pd.concat(
        {
            name: report.set_index("step")["remaining"]
            for name, report in reports.items()
        },
        axis=1,
    )
)

In [ ]:
fig, axes = plt.subplots(
    1,
    len(reports),
    figsize=(6.5 * len(reports), 0.45 * len(next(iter(reports.values()))) + 2),
    sharex=False,
)

for ax, (name, report) in zip(axes, reports.items()):
    enrolment_plots.enrolment_funnel(report, title=name, ax=ax)

fig.tight_layout()

os.makedirs(figures_folder, exist_ok=True)
discharge_group_counts = {
    name: {
        "nursing_ward": int(
            frame["discharge_care_level"].ne("high_dependency_unit").sum()
        ),
        "high_dependency_unit": int(
            frame["discharge_care_level"].eq("high_dependency_unit").sum()
        ),
    }
    for name, frame in enrolled.items()
}
enrolment_svg = enrolment_plots.enrolment_flow_svg(
    reports, discharge_group_counts=discharge_group_counts
)
enrolment_svg_path = os.path.join(figures_folder, "Enrolment.svg")
with open(enrolment_svg_path, "w", encoding="utf-8", newline="\n") as svg_file:
    svg_file.write(enrolment_svg)
print(f"Wrote enrolment diagram to {enrolment_svg_path}")
display(SVG(data=enrolment_svg))

In [ ]:
# how many stays each step removed, per cohort
pd.concat(
    {name: report.set_index("step")["excluded"] for name, report in reports.items()},
    axis=1,
).drop(index="total_icu_admissions")

## Enrolled cohort

Contains stays that meet the outcome definition and have both physician and model predictions.


In [ ]:
enrolment_summary = pd.DataFrame(
    [
        {
            "cohort": name,
            "screened": len(enrolment_cohorts[name]),
            "enrolled": len(frame),
            "unique_patients": len(frame[["hospital", "patient_id"]].drop_duplicates()),
            "events": int(frame["outcome"].fillna(False).sum()),
            "enrolled_pct": round(100 * len(frame) / len(enrolment_cohorts[name]), 1),
        }
        for name, frame in enrolled.items()
    ]
)
print(enrolment_summary)
event_summary = pd.DataFrame(
    [
        {
            "cohort": name,
            "icu_readmissions_7d": int(
                frame["has_icu_readmission_7days"].fillna(False).sum()
            ),
            "deaths_7d": int(frame["has_death_7days"].fillna(False).sum()),
            "composite_events": int(frame["outcome"].fillna(False).sum()),
            "event_patients": len(
                frame.loc[
                    frame["outcome"].fillna(False),
                    ["hospital", "patient_id"],
                ].drop_duplicates()
            ),
        }
        for name, frame in enrolled.items()
    ]
)
print(event_summary)
event_combinations = (
    enrolled["Combined"]
    .assign(
        icu_readmission=lambda df: df["has_icu_readmission_7days"].fillna(False),
        death=lambda df: df["has_death_7days"].fillna(False),
    )
    .groupby(["icu_readmission", "death"], dropna=False)
    .size()
    .rename("stays")
    .reset_index()
)
print(event_combinations[event_combinations[["icu_readmission", "death"]].any(axis=1)])

# Validate the analysis cohort before downstream tables and performance estimates.
for name in cohorts.HOSPITALS:
    screened = enrolment_cohorts[name]
    analysed = enrolled[name]
    assert (
        not screened["icu_stay_id"].duplicated().any()
    ), f"{name}: duplicate ICU-stay identifiers in the screened cohort"
    assert (
        analysed["discharge_destination"].eq("ward").all()
    ), f"{name}: final cohort contains an ineligible discharge destination"
    assert (
        analysed["discharge_care_level"]
        .isin(["general_ward", "high_dependency_unit", "unknown"])
        .all()
    ), f"{name}: final cohort has an unknown discharge care-level value"
    assert not (
        analysed["palliative_care"].fillna(False)
        | analysed["do_not_readmit"].fillna(False)
    ).any(), f"{name}: final cohort contains a treatment limitation"

vumc_predictions = enrolment_cohorts["VUmc"].dropna(subset=["doc_predicted_at"]).copy()
for column in ("doc_predicted_at", "icu_admitted_at", "icu_discharged_at"):
    vumc_predictions[column] = pd.to_datetime(vumc_predictions[column])
vumc_prediction_within_stay = (
    vumc_predictions["doc_predicted_at"] >= vumc_predictions["icu_admitted_at"]
) & (
    vumc_predictions["doc_predicted_at"]
    <= vumc_predictions["icu_discharged_at"] + pd.Timedelta(hours=24)
)
assert (
    vumc_prediction_within_stay.all()
), "VUmc: physician prediction outside the ICU-stay boundary"
print("Cohort integrity checks passed")

## Prediction availability

Compare availability separately for physicians and the algorithm after the seven outcome-eligibility exclusions. Algorithm unavailability includes predictions withheld after OOD classification. No prediction-completeness exclusion has been applied at this point.


In [ ]:
from algo_vs_doc import comparison

physician_availability = {
    name: comparison.prediction_source_availability(
        frame, column="doc_predicted_at", source="physician"
    )
    for name, frame in outcome_eligible.items()
}
algorithm_availability = {
    name: comparison.prediction_source_availability(
        frame, column="algo_predicted_at_doc_prediction", source="algorithm"
    )
    for name, frame in outcome_eligible.items()
}

pd.DataFrame(
    {
        (name, source): values.value_counts().reindex(
            comparison.PREDICTION_PAIR_AVAILABILITY
        )
        for name in outcome_eligible
        for source, values in (
            ("physician", physician_availability[name]),
            ("algorithm", algorithm_availability[name]),
        )
    }
).rename_axis("predictions available")

### Characteristics by physician prediction availability

Report numeric variables as median [IQR] and binary variables as n (%). These hospital-specific comparisons are descriptive; no row-level hypothesis tests are performed.


In [ ]:
# Keep descriptive availability comparisons aligned with the clustered models.
NUMERIC = list(
    dict.fromkeys(
        [
            "icu_los",
            "sofa_admission",
            "sofa_max",
            "news_discharge",
            "discharge_local_hour",
            "new_diagnoses_count",
            *clustered_inference.CASE_MIX_VARIABLES,
            *(
                variable
                for variable in clustered_inference.PRESSURE_VARIABLES
                if variable not in clustered_inference.BOOLEAN_VARIABLES
            ),
        ]
    )
)
BINARY = list(
    dict.fromkeys(
        [
            *clustered_inference.ADMISSION_TYPE_COLUMNS,
            *(
                variable
                for variable in clustered_inference.PRESSURE_VARIABLES
                if variable in clustered_inference.BOOLEAN_VARIABLES
            ),
            "ventilation",
            "invasive_ventilation",
            "renal_replacement_therapy",
            "vasopressors_inotropes",
            "outcome",
        ]
    )
)

### Comparison by hospital

Compare characteristics according to physician prediction availability separately within each hospital.


In [ ]:
availability_characteristic_tables = []
for name in cohorts.HOSPITALS:
    print(f"=== {name}: Physician prediction ===")
    table = comparison.compare_groups(
        outcome_eligible[name],
        physician_availability[name],
        NUMERIC + BINARY,
        binary=BINARY,
        descriptive_only=NUMERIC + BINARY,
    )
    display(table)
    group_columns = [
        column
        for column in table.columns
        if column not in {"variable", "test", "p", "p_holm"}
    ]
    if len(group_columns) != 2:
        raise ValueError(f"Expected two physician-availability groups for {name}")
    availability_characteristic_tables.append(
        table.rename(
            columns={
                group_columns[0]: "prediction_available",
                group_columns[1]: "prediction_missing",
            }
        )
        .loc[:, ["variable", "prediction_available", "prediction_missing"]]
        .assign(cohort=name)
    )

pd.concat(availability_characteristic_tables, ignore_index=True).to_csv(
    os.path.join(
        outputs_folder, "physician_prediction_availability_characteristics.csv"
    ),
    index=False,
)

## Exploratory analysis of physician prediction availability

Separate logistic regression models for VUmc and LUMC estimate associations between measured operational and case-mix characteristics and physician prediction availability. Standard errors are clustered by patient and hospital-local discharge date. Results are reported as adjusted odds ratios with 95% confidence intervals without hypothesis-test p-values.


In [ ]:
physician_availability_models = pd.concat(
    [
        clustered_inference.prediction_availability_model(
            outcome_eligible[name], prediction_column="doc_predicted_at"
        ).assign(cohort=name, prediction_source="physician")
        for name in cohorts.HOSPITALS
    ],
    ignore_index=True,
)
availability_denominators = pd.DataFrame(
    [
        {
            "cohort": name,
            "outcome_eligible_stays": len(frame),
            "physician_prediction_available": int(
                frame["doc_predicted_at"].notna().sum()
            ),
            "physician_prediction_missing": int(frame["doc_predicted_at"].isna().sum()),
            "algorithm_prediction_available": int(
                frame["algo_predicted_at_doc_prediction"].notna().sum()
            ),
            "algorithm_prediction_missing": int(
                frame["algo_predicted_at_doc_prediction"].isna().sum()
            ),
            "paired_predictions_available": int(
                (
                    frame["doc_predicted_at"].notna()
                    & frame["algo_predicted_at_doc_prediction"].notna()
                ).sum()
            ),
            "hospital_discharge_dates": int(
                frame[["hospital"]]
                .assign(
                    discharge_date=local_time.amsterdam(
                        frame["icu_discharged_at"]
                    ).dt.date
                )
                .drop_duplicates()
                .shape[0]
            ),
        }
        for name, frame in outcome_eligible.items()
    ]
)
physician_availability_models.to_csv(
    os.path.join(
        outputs_folder, "hospital_specific_physician_prediction_availability_models.csv"
    ),
    index=False,
)
availability_denominators.to_csv(
    os.path.join(outputs_folder, "prediction_availability_denominators.csv"),
    index=False,
)
display(availability_denominators)
display(physician_availability_models)